# Финальное обучение: Baseline++ с 6 слоями Encoder и 7 слоями Decoder (aka 67) - Resume

Надо скопировать этот ноутбук и выполнить общие ячейки, затем выбрать **одну из трёх веток**: с нуля, с весов или resume. У каждой ветки свои параметры, сетап и запуск. Для последовательного запуска нескольких новых экспериментов предусмотрен отдельный блок очереди в конце; он выполняется вместо одиночной ветки.

Обучение запускается на двух GPU T4 в Kaggle. Метрики отправляются в W&B, результаты сохраняются в `/kaggle/working/experiment_runs`.

## Общие пути

- `PROJECT_PATH`: репозиторий с `utilities/train_script.py` и конфигами.
- `DATA_PATH`: исходная папка `tokenized` с `dataset_dict.json`, `tokenizer.json`, `train` и `val`. Используется при создании нового эксперимента и подготовке отсутствующих данных для resume.
- `WORK_DIR`: рабочая папка Kaggle для конфигов, данных и результатов.

Имя эксперимента и имя конфига задаются отдельно в ветках 1 и 2. Начальные веса нужны только ветке 2, папка сохранённого эксперимента — только ветке 3.

In [1]:
from pathlib import Path

PROJECT_PATH = Path("/kaggle/input/datasets/wwip62/nlp-hw2-codegen").expanduser().resolve()
DATA_PATH = Path("/kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized").expanduser().resolve()
WORK_DIR = Path("/kaggle/working").resolve()

TOKENIZER_PATH = DATA_PATH / "tokenizer.json"
DATA_DIR = WORK_DIR / "tokenized"

In [2]:
if not (PROJECT_PATH / "utilities" / "train_script.py").is_file():
    raise FileNotFoundError(f"Скрипт обучения не найден в репозитории: {PROJECT_PATH}")

print("Репозиторий:", PROJECT_PATH)
print("Исходные данные:", DATA_PATH)
print("Рабочая папка:", WORK_DIR)

Репозиторий: /kaggle/input/datasets/wwip62/nlp-hw2-codegen
Исходные данные: /kaggle/input/datasets/wwip62/codegen-tokenized-bpe-16384/tokenized
Рабочая папка: /kaggle/working


In [3]:
%pip install -q datasets tokenizers tqdm PyYAML "sacrebleu>=2.4,<3" wandb

Note: you may need to restart the kernel to use updated packages.


## Импорты

In [4]:
import os
import shutil
import subprocess
import sys

import torch
import yaml
from kaggle_secrets import UserSecretsClient

## Общие настройки

В Kaggle надо включить **GPU T4 ×2**. `NUM_GPUS` и W&B используются во всех ветках. Параметры LR и warmup применяются только к новым экспериментам в ветках 1 и 2. Resume восстанавливает настройки из сохранённого эксперимента.

In [5]:
NUM_GPUS = 2

EXTRA_TRAIN_SECONDS = 5 * 60 * 60

WANDB_PROJECT = "nlp-hw2"
WANDB_ENTITY = "62wip-hse-ml"


## Настройка W&B

In [6]:
env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["WANDB_MODE"] = "online"
# Надо проставить WANDB_API_KEY в Kaggle Secrets
env["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Подготовка данных и нового эксперимента

`prepare_data` копирует train и валидацию в рабочую папку `/kaggle/working/tokenized`: обучение читает локальную копию, а кеш валидации записывается рядом с данными. Копирование train требует около 8 ГиБ диска и времени перед запуском. Функция вызывается при новом эксперименте и при отсутствии данных или старой ссылке на train для resume.

`prepare_train_config` используется только в ветках 1 и 2: создаёт конфиг с абсолютными путями и именем эксперимента. Параметры warmup переопределяются независимо; `None` сохраняет соответствующее значение исходного конфига.

In [7]:
def prepare_data(data_dir=DATA_DIR):
    dataset_file = DATA_PATH / "dataset_dict.json"
    if not dataset_file.is_file():
        raise FileNotFoundError(f"Файл данных не найден: {dataset_file}")
    for split in ("train", "val"):
        if not (DATA_PATH / split).is_dir():
            raise FileNotFoundError(f"Выборка не найдена: {DATA_PATH / split}")

    data_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DATA_PATH / "dataset_dict.json", data_dir / "dataset_dict.json")
    shutil.copytree(DATA_PATH / "val", data_dir / "val", dirs_exist_ok=True)

    train_path = data_dir / "train"
    if train_path.is_symlink():
        train_path.unlink()
    shutil.copytree(DATA_PATH / "train", train_path, dirs_exist_ok=True)


## Resume

Продолжение существующего эксперимента из `checkpoints/last.pt`: восстанавливаются модель, оптимизатор, scheduler и прогресс. Надо явно задать полный путь `PATH_TO_EXPERIMENT` к папке эксперимента, доступной для записи. Resume использует `last.pt`.

Новый конфиг, `RUN_NAME`, `CONFIG_NAME` и `WEIGHTS_FROM` не нужны. Токенайзер берётся из `init_state/tokenizer.json`, данные — по пути из сохранённого `config.yaml`. Если данных по сохранённому пути ещё нет, они подготавливаются из `DATA_PATH`. Исходный токенайзер для этого не требуется. Число GPU должно совпадать с сохранённым запуском. При наличии `init_state/wandb.json` продолжается тот же запуск W&B.

In [8]:
PATH_TO_EXPERIMENT = "/kaggle/input/datasets/wwip62/output-baseline-67-11h/experiment_runs/mha_6enc_7dec"

if PATH_TO_EXPERIMENT is ...:
    raise ValueError("Надо задать PATH_TO_EXPERIMENT: полный путь к папке эксперимента для продолжения.")

EXPERIMENT_SOURCE = Path(PATH_TO_EXPERIMENT).expanduser().resolve()
RESUME_PATH = WORK_DIR / "experiment_runs" / EXPERIMENT_SOURCE.name

if not RESUME_PATH.exists():
    RESUME_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(EXPERIMENT_SOURCE, RESUME_PATH)

print("Исходный эксперимент:", EXPERIMENT_SOURCE)
print("Рабочая копия для resume:", RESUME_PATH)

Исходный эксперимент: /kaggle/input/datasets/wwip62/output-baseline-67-11h/experiment_runs/mha_6enc_7dec
Рабочая копия для resume: /kaggle/working/experiment_runs/mha_6enc_7dec


In [9]:
required_resume_files = [
    RESUME_PATH / "config.yaml",
    RESUME_PATH / "init_state" / "tokenizer.json",
    RESUME_PATH / "init_state" / "sample_indices.json",
    RESUME_PATH / "checkpoints" / "last.pt",
]
missing_files = [str(path) for path in required_resume_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError("Некорректный RESUME_PATH. Не найдены файлы:\n" + "\n".join(missing_files))


resume_config = yaml.safe_load((RESUME_PATH / "config.yaml").read_text())
resume_data_dir = (WORK_DIR / Path(resume_config["data"]["tokenized_dir"])).resolve()
if (
    not (resume_data_dir / "dataset_dict.json").is_file()
    or (resume_data_dir / "train").is_symlink()
    or not all((resume_data_dir / split).is_dir() for split in ("train", "val"))
):
    prepare_data(resume_data_dir)

print("Продолжение:", RESUME_PATH)
print("Данные:", resume_data_dir)

Продолжение: /kaggle/working/experiment_runs/mha_6enc_7dec
Данные: /kaggle/working/tokenized


## Исходный scheduler и таймер продолжения

После копирования **всей папки эксперимента** в доступный для записи `PATH_TO_EXPERIMENT` выполняется обычный `--resume` из `checkpoints/last.pt`. Общие пути сохранены из `train_example`. Число GPU должно совпадать с исходным запуском.

Продолжаются сохранённые веса, AdamW, scaler, RNG, прогресс, история и исходный scheduler. В итоговой истории последний LR равен примерно `7.58e-5`; resume берёт фактическое значение из checkpoint и продолжает его снижение. Плато, повышение LR и увеличение `num_epochs` не задаются.

`EXTRA_TRAIN_SECONDS` оставляет **максимум пять дополнительных часов**. Обучение заканчивается при достижении исходного числа эпох или лимита времени — в зависимости от того, что наступит раньше. Для присланной истории до конца исходных двух эпох остаётся ориентировочно 3.8 часа при прежней скорости. Ячейка ниже только читает checkpoint и рассчитывает общий лимит с учётом отработанного времени. Подготовка данных происходит до таймера; финальная валидация и сохранение могут закончиться немного позже лимита.

Существующий `best.pt` и исходный `init_state/wandb.json` используются обычным механизмом resume. Если прежняя ячейка с плато уже выполнялась, для этого варианта нужна исходная папка эксперимента или согласованная пара исходных checkpoint и конфига из `before_resume_lr`.


In [10]:
from statistics import median

checkpoint_path = RESUME_PATH / "checkpoints" / "last.pt"
saved = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
if saved.get("process_count", 1) != NUM_GPUS:
    raise ValueError("NUM_GPUS должен совпадать с process_count сохранённого запуска.")

previous_seconds = saved["details"].get(
    "elapsed_seconds",
    sum(row["train_seconds"] + row.get("val_seconds", 0.0) for row in saved["validation_history"]),
)
LIMIT_TIME = previous_seconds + EXTRA_TRAIN_SECONDS

print("Продолжение исходного scheduler:", resume_config.get("scheduler"))
print("LR из checkpoint:", [group["lr"] for group in saved["optimizer_state_dict"]["param_groups"]])
print(f"Эпох завершено: {len(saved['history'])}; исходная цель: {resume_config['training']['num_epochs']}")
print(f"Дополнительный лимит: {EXTRA_TRAIN_SECONDS / 3600:g} ч; общий --max-train-seconds {LIMIT_TIME:.1f} с")

interval = resume_config["training"]["examples_per_mini_epoch"]
train_size = resume_config["data"]["train_size"]
full_rows = [
    row for row in saved["validation_history"]
    if interval and row.get("mini_epoch_examples", 0) >= 0.9 * interval
][-5:]
if full_rows and train_size is not None:
    remaining_examples = max(
        0, resume_config["training"]["num_epochs"] * train_size - saved["progress"]["examples_seen"]
    )
    seconds_per_example = median(
        (row["train_seconds"] + row.get("val_seconds", 0.0)) / row["mini_epoch_examples"]
        for row in full_rows
    )
    print(f"До исходного конца эпох: ~{remaining_examples * seconds_per_example / 3600:.2f} ч при прежней скорости")

del saved


Продолжение исходного scheduler: {'type': 'CosineAnnealingLR', 'params': {'T_max': 111, 'eta_min': 3e-05}, 'warmup': {'steps': 5, 'start_factor': 0.1}}
LR из checkpoint: [7.580891234233914e-05]
Эпох завершено: 1; исходная цель: 2
Дополнительный лимит: 5 ч; общий --max-train-seconds 57695.0 с
До исходного конца эпох: ~3.78 ч при прежней скорости


In [11]:
assert torch.cuda.device_count() >= NUM_GPUS, "Для запуска требуется GPU T4 ×2 в настройках Kaggle."

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={NUM_GPUS}",
    "-m", "utilities.train_script",
    "--resume", str(RESUME_PATH),
    "--max-train-seconds", str(LIMIT_TIME),
    "--wandb-project", WANDB_PROJECT,
]

if WANDB_ENTITY:
    command += ["--wandb-entity", WANDB_ENTITY]

subprocess.run(
    command,
    cwd=PROJECT_PATH,
    env={**env, "PROJECT_DIR": str(WORK_DIR)},
    check=True,
)


*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
Flattening the indices: 100%|██████████| 1024/1024 [00:00<00:00, 42923.06 examples/s]


Run: /kaggle/working/experiment_runs/mha_6enc_7dec; device: cuda:0; processes: 2; max_tokens per process: 8192


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: 62wip (62wip-hse-ml) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/experiment_runs/mha_6enc_7dec/wandb/run-20261024_132518-h6vjyo5w
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run mha_6enc_7dec
wandb: ⭐️ View project at https://wandb.ai/62wip-hse-ml/nlp-hw2
wandb: 🚀 View run at https://wandb.ai/62wip-hse-ml/nlp-hw2/runs/h6vjyo5w


Epoch 2, mini-epoch 30 | train loss = 0.2138, val loss = 0.2043
Examples: 17,310,602 | time = 443.8 s (train 357.8, val 85.9) | train/example = 0.001923 s
Small validation: token accuracy = 93.60%, top-5 = 99.32%
Heavy validation: pass@1 = 69.34% (710 / 1024) | time = 77.4 s
Syntax valid = 99.12%, numeric return = 96.97%, no EOS = 2.73%
BLEU = 57.15, repetition = 7.29%, timeout = 0.00%
Epoch 2, mini-epoch 31 | train loss = 0.2122, val loss = 0.2038
Examples: 17,510,510 | time = 466.0 s (train 380.1, val 86.0) | train/example = 0.001901 s
Small validation: token accuracy = 93.61%, top-5 = 99.32%
Heavy validation: pass@1 = 69.43% (711 / 1024) | time = 77.6 s
Syntax valid = 98.93%, numeric return = 96.68%, no EOS = 3.61%
BLEU = 57.75, repetition = 7.32%, timeout = 0.00%
Epoch 2, mini-epoch 32 | train loss = 0.2111, val loss = 0.2035
Examples: 17,710,589 | time = 463.6 s (train 378.7, val 84.9) | train/example = 0.001893 s
Small validation: token accuracy = 93.61%, top-5 = 99.32%
Heavy val

wandb: updating run metadata
wandb: uploading wandb-summary.json


CompletedProcess(args=['/usr/bin/python3', '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'utilities.train_script', '--resume', '/kaggle/working/experiment_runs/mha_6enc_7dec', '--max-train-seconds', '57694.976291203', '--wandb-project', 'nlp-hw2', '--wandb-entity', '62wip-hse-ml'], returncode=0)

## 5. Удаление рабочей копии данных

Эта ячейка удаляет `/kaggle/working/tokenized` вместе с кешем данных. Чекпойнты и метрики в `experiment_runs` сохраняются. При следующем запуске или resume данные будут скопированы заново.

In [12]:
cleanup_dir = WORK_DIR / "tokenized"
if cleanup_dir.is_symlink():
    cleanup_dir.unlink()
elif cleanup_dir.exists():
    shutil.rmtree(cleanup_dir)
print("Рабочая копия данных удалена:", cleanup_dir)

Рабочая копия данных удалена: /kaggle/working/tokenized
